In [23]:
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import pickle
import torch

In [ ]:
import os
os.getcwd()
os.chdir('../..')
os.getcwd()

'/Users/nkapila6/Code/nlpvise/src'

In [8]:
with open("data/embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

In [9]:
embeddings[0:3]

array([[-0.03600806, -0.05763245, -0.5764836 , ...,  0.05967164,
        -0.06377713, -0.15175182],
       [ 0.04741407, -0.25133044, -0.42309144, ...,  0.02087078,
         0.12380733, -0.0428645 ],
       [ 0.0549481 ,  0.30195963, -0.5842947 , ..., -0.05654861,
         0.12264945, -0.35312414]], dtype=float32)

In [10]:
notes = pd.read_pickle('data/notes_with_embeddings.pkl')

In [12]:
notes.columns

Index(['Unnamed: 0', 'ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CHARTDATE',
       'CHARTTIME', 'STORETIME', 'CATEGORY', 'DESCRIPTION', 'CGID', 'ISERROR',
       'TEXT', 'PTEXT', 'FTEXT', 'SENT', 'EMBEDDING'],
      dtype='object')

In [15]:
sent = notes['SENT'][0]

In [16]:
sent

'titl respiratori failur acut doctor last name assess resp statu stabl psv chang sat rr continu cough freq ask sx freq less previous exp wheez lung field consist noc improv mdi pt pain throat sore face ett action sx pt minim thin secret lidocain ett ask pt take ativan morphin help symptom pt took 1mg ativan 12 hr shift ett tube retap rotat pt like tube l side mouth examin intact respons lidocain w relief cough vari period time relief pain throat retap tube pt abl sleep hour plan cont monitor resp statu sat rr vent paramet pt comfort cont offer medic lidocain prn mdi q4hr ineffect cope assess pt cont control amount sed medic minimum appear afraid die famili consist close bedsid sister encourag pt take med particularli noc get sleep reliev symptom pt want sedat pt unwil get extub thu far afraid die action encourag pt take med symptom respons plan'

In [18]:
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = AutoModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
model = model.to('mps')
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [19]:
inp = tokenizer(sent, return_tensors='pt', truncation=True, max_length=512, padding=True)

In [20]:
inp = {k:v.to('mps') for k,v in inp.items()}

In [24]:
with torch.no_grad():
    o = model(**inp)

In [25]:
e = o.last_hidden_state[:, 0, :].cpu().numpy()[0]

In [27]:
np.array_equal(e, embeddings[0])

True

In [28]:
np.array_equal(e, notes['EMBEDDING'][0])

True